# 02 — Visualizing LCZ Maps and Quantifying Class Areas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/02_visualization_area_stats.en.ipynb)

**Learning objective**: learn to turn a raw LCZ classification raster into two things planners and reviewers actually need — an interactive, zoomable map anyone can explore in a browser, and a quantitative breakdown of how much area each LCZ class occupies. By the end of this notebook you will be able to render an LCZ map with `lcz_plot_map`, switch between discrete-class and continuous-raster visualization modes, and compute/chart per-class areas with `lcz_cal_area` using five different chart styles.

## Why interactive visualization matters for LCZ science communication

The Local Climate Zone scheme (Stewart & Oke, 2012) was designed as a *standardized*, worldwide-comparable classification of urban and natural landscapes into 17 classes (1–10 built types, from compact high-rise to large low-rise; A–G / 11–17 natural land covers, from dense trees to water). A classification is only useful, though, if the people who need to act on it — urban planners, city councils, public-health officers, climate-adaptation teams — can actually read it. A static PNG buried in a PDF report rarely survives the trip from the researcher's desktop to the planning meeting; an interactive map that a stakeholder can pan, zoom, and hover over to see the exact class at a given street corner does.

`lcz_plot_map` (LCZ4py, following the visualization philosophy of the LCZ4r R package, Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) renders LCZ rasters as Plotly WebGL figures — self-contained HTML that opens in any browser, supports hover tooltips, zoom/pan, and (via the `renderer='maplibre'` option) can overlay the classification on a real OpenStreetMap basemap for immediate geographic context.

**Discrete vs. continuous rasters.** Two fundamentally different kinds of raster show up in LCZ workflows, and `lcz_plot_map` handles both:

- **Discrete-class choropleth**: the LCZ map itself — every pixel holds an integer class code (1–17). The right way to render this is a categorical color map with one flat color per class and a legend listing class names, *not* a continuous color gradient (a gradient would visually imply that class 5 is 'between' classes 4 and 6, which is meaningless — the LCZ codes are a nominal, not ordinal, scale).
- **Continuous raster overlay**: derived physical fields on the same grid, e.g. per-pixel Land Surface Temperature (LST) from `lcz_get_lst`, or a morphological parameter from `lcz_get_parameters`. These *are* ordinal/interval data, so a diverging or sequential colorscale (e.g. `RdBu_r`, blue=cold to red=hot) with a colorbar is the correct rendering, and averaging or interpolating between pixel values is meaningful.

`lcz_plot_map`'s `data_type` parameter (`'auto'`, `'lcz'`, `'continuous'`) picks the right mode automatically by inspecting the raster's dtype, but you can force it explicitly. Mixing the two up — e.g. applying a categorical palette to a temperature raster — is a common and avoidable source of misleading urban-climate figures.

**Why area quantification comes right after visualization.** A map answers *where*; a bar/pie/donut/sunburst/treemap of per-class area answers *how much*. Before any deeper urban-climate analysis (UHI intensity, thermal comfort modeling, morphological parameter extraction), it is standard practice to first ask: what fraction of this city is compact high-rise (LCZ 1) vs. open low-rise (LCZ 6) vs. dense trees (LCZ A/11) vs. water (LCZ G/17)? `lcz_cal_area` computes exact per-class area in km² (accounting for the raster's actual pixel size and CRS) and renders it as one of five interactive chart types, each suited to a different communication need.

## Install LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Get a real LCZ map to work with

We use Vaduz (Liechtenstein) — a small city whose bounding box keeps the download fast — so this notebook runs end-to-end in a couple of minutes. Everything below applies unchanged to any city.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Vaduz")
print(map_path)

06:25:44 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


## `lcz_plot_map` — interactive discrete-class map

Key parameters:
- `x`: path to the LCZ GeoTIFF (or an already-open rasterio dataset).
- `show_legend`: shows the 17-class legend (recommended, on by default).
- `inclusive`: swaps the standard LCZ palette for a colorblind-friendly one (see below).
- `use_webgl` / `max_pixels`: keep large rasters responsive by rendering via WebGL and down-sampling above a pixel threshold.
- `renderer`: `'plotly'` (self-contained figure, default) or `'maplibre'` (HTML page with an OpenStreetMap basemap underneath — good for giving stakeholders geographic bearings).
- `data_type`: `'auto'` (default) detects LCZ vs. continuous data from the raster dtype.

It returns an `LCZPlotResult` with a `.fig` (Plotly figure) and, for `maplibre`, `.html`. `result.show()` displays either.

In [3]:
from LCZ4py import lcz_plot_map

result = lcz_plot_map(map_path, show_legend=True, title="LCZ map — Vaduz")
result.fig.show()

**Reading this map**: each flat-colored patch is one LCZ class — warm reds/oranges for the built types (compact/open low- or mid-rise are typical for a small alpine town like Vaduz), greens for vegetated classes (dense/scattered trees on the surrounding slopes), and blue for water/river. Hover over any pixel to see its exact class in the tooltip. For a town this size, expect the built footprint to be a small fraction of the total mapped area, with natural classes (forest, low plants, bare rock) dominating — a pattern very different from a dense metropolitan LCZ map.

## The colorblind-friendly palette (`inclusive=True`)

The LCZ scheme's standard palette (Stewart & Oke 2012) uses reds/oranges for built classes and greens/blues for natural ones — a scheme that is hard or impossible to distinguish for the ~4-8% of readers with red-green color vision deficiency (the most common form of color blindness). Setting `inclusive=True` swaps in a perceptually distinct, colorblind-safe palette that preserves the same class boundaries and legend. This costs nothing in the code (one keyword) and matters a great deal for the accessibility of published figures, conference posters, and any material shared with a general audience — accessible science communication should be the default, not an afterthought reserved for readers who explicitly ask.

In [4]:
result_inclusive = lcz_plot_map(map_path, show_legend=True, inclusive=True,
                                title="LCZ map — Vaduz (colorblind-friendly palette)")
result_inclusive.fig.show()

Compare the two figures above: the class boundaries are identical, only the color encoding changed. In any report or paper intended for a broad audience, prefer `inclusive=True`.

## Per-class area statistics with `lcz_cal_area`

Key parameters:
- `x`: path to the LCZ GeoTIFF.
- `plot_type`: `'bar'`, `'pie'`, `'donut'`, `'sunburst'`, or `'treemap'`.
- `iplot`: if `False`, skip chart rendering and return only the area `DataFrame`.
- `inclusive`: colorblind-friendly palette, same as `lcz_plot_map`.
- `use_duckdb` / `use_geoarrow`: optional accelerated backends for large rasters (degrade gracefully to Polars-only if not installed).

Returns an `LCZAreaResult` with `.df` (a Polars DataFrame: LCZ class, name, area in km², percent of mapped area), `.fig` (the Plotly chart), and `.geoarrow_table` (optional zero-copy polygon export). This is the standard first quantitative step in any LCZ-based urban climate study: before computing UHI intensity or morphological parameters, you want to know what fraction of the city is urban vs. natural land cover.

### Bar chart — best for precise class-by-class comparison

A bar chart is the right default whenever the audience needs to compare exact magnitudes across all 17 classes, or rank classes by area. Bars make small differences between adjacent classes easy to read off, which pie-style charts do not.

In [5]:
from LCZ4py import lcz_cal_area

area_bar = lcz_cal_area(map_path, plot_type="bar", title="LCZ class area — Vaduz")
area_bar.fig.show()
area_bar.df

lcz,count,area_km2,area_perc,lcz_name,lcz_col,lcz_colorblind
i64,i64,f64,f64,str,str,str
5,101,0.69,3.98,"""Open midrise""","""#FF6628""","""#989600"""
6,306,2.09,12.05,"""Open lowrise""","""#FF985E""","""#739F00"""
8,22,0.15,0.87,"""Large lowrise""","""#BBBBBB""","""#00AA63"""
9,119,0.81,4.69,"""Sparsely built""","""#FFCBAB""","""#00AD89"""
11,1260,8.6,49.63,"""Dense trees""","""#006A18""","""#00A7C5"""
12,88,0.6,3.47,"""Scattered trees""","""#00A926""","""#009EDA"""
14,244,1.66,9.6,"""Low plants""","""#B5DA7F""","""#9E7FE5"""
15,389,2.66,15.33,"""Bare rock or paved""","""#000000""","""#C36FDA"""
17,10,0.07,0.39,"""Water""","""#656BFA""","""#E264A9"""


**Reading this**: the tallest bars identify the dominant land cover(s) for the study area. For Vaduz, expect one or two natural classes (forest/dense trees, low plants) to dominate, with each built class contributing a comparatively small share — quantitative confirmation of what the map showed visually above.

### Donut chart — best for a quick 'urban vs. natural' share at a glance

A donut (or pie) chart is preferable when the message is a *proportion* story — e.g. "what fraction of the city is built-up?" — rather than a precise ranking. The visual whole-to-part framing communicates that immediately to a non-technical audience (city council, press release), at the cost of making small-class comparisons harder than a bar chart would.

In [6]:
area_donut = lcz_cal_area(map_path, plot_type="donut", inclusive=True,
                          title="LCZ class area share — Vaduz")
area_donut.fig.show()

**Reading this**: each slice's angular size is proportional to its area share. Grouping the slices mentally into 'warm-colored' (built, LCZ 1-10) vs. 'cool-colored' (natural, LCZ A-G/11-17) gives an instant read on the urban-vs-natural land-cover balance — exactly the number a planner asks for first.

### Treemap — best for showing hierarchy and relative area with nested detail

A treemap encodes area share as rectangle size (like a donut) but also naturally supports grouping classes into a parent/child hierarchy (e.g. "Built" and "Natural" as parent groups, individual LCZ classes nested inside). It is the preferable choice when you want both the whole-to-part story *and* a compact, information-dense layout that also legibly shows every class's exact label and value in the same figure — useful in a written report where a full page is available but you don't want a wall of bars.

In [7]:
area_treemap = lcz_cal_area(map_path, plot_type="treemap",
                            title="LCZ class area — Vaduz (treemap)")
area_treemap.fig.show()

**Reading this**: rectangle area is proportional to each class's mapped area; larger rectangles are the dominant classes. Unlike the bar chart, small classes are still visible (as small tiles) rather than compressed against an axis, which is handy when you need every class labeled in one static export for a report figure.

### Getting just the numbers (no chart)

Set `iplot=False` when you only need the area table — e.g. to feed into a downstream statistical analysis or a custom figure elsewhere in a pipeline.

In [8]:
area_df = lcz_cal_area(map_path, iplot=False)
area_df

lcz,count,area_km2,area_perc,lcz_name,lcz_col,lcz_colorblind
i64,i64,f64,f64,str,str,str
5,101,0.69,3.98,"""Open midrise""","""#FF6628""","""#989600"""
6,306,2.09,12.05,"""Open lowrise""","""#FF985E""","""#739F00"""
8,22,0.15,0.87,"""Large lowrise""","""#BBBBBB""","""#00AA63"""
9,119,0.81,4.69,"""Sparsely built""","""#FFCBAB""","""#00AD89"""
11,1260,8.6,49.63,"""Dense trees""","""#006A18""","""#00A7C5"""
12,88,0.6,3.47,"""Scattered trees""","""#00A926""","""#009EDA"""
14,244,1.66,9.6,"""Low plants""","""#B5DA7F""","""#9E7FE5"""
15,389,2.66,15.33,"""Bare rock or paved""","""#000000""","""#C36FDA"""
17,10,0.07,0.39,"""Water""","""#656BFA""","""#E264A9"""


## Conclusion

You now have two complementary tools for communicating an LCZ classification: `lcz_plot_map` for an interactive, explorable spatial view (with a colorblind-friendly option via `inclusive=True`, and both a discrete-class mode for LCZ maps and a continuous mode for rasters like LST), and `lcz_cal_area` for precise per-class area quantification across five chart styles (bar for precise comparison, donut/pie for quick proportion stories, sunburst/treemap for hierarchical or information-dense layouts). Together they form the standard first deliverable of any LCZ-based urban climate study: *where* the classes are, and *how much* of the city each one covers.

**Previous**: `01_map_acquisition` (downloading and sourcing LCZ maps).
**Next**: `03_morphological_parameters` — extracting the 34 morphological/thermal parameters (building surface fraction, sky view factor, roughness, etc.) that underlie each LCZ class.